# Fine-Tuning CatBoost on Mixed Categorical + Numeric Data

A practical, end-to-end guide to building, tuning, and interpreting a **CatBoostClassifier** on your `churn_train_cleaned.csv` dataset.

**What this notebook covers**

1. Environment setup and dependency install
2. Loading and inspecting the churn dataset
3. Declaring feature types (categorical / numeric / text) — and why this matters more in CatBoost than in any other GBDT
4. A clean baseline model with proper validation
5. **Highlighting specific columns to the model** via two complementary mechanisms:
   - `feature_weights` — bias the *split selection* toward columns you care about
   - `per_float_feature_quantization` — give important numeric columns *finer-grained* bin boundaries
6. Class-imbalance handling (your data is roughly balanced, but every real churn problem drifts — included so you have it for production)
7. A full Optuna hyperparameter-tuning deep-dive
8. SHAP interpretation — the critical check that `feature_weights` is doing what you intended
9. Saving / loading / extending to multiclass

> **Note on the target.** Your `Churned` column is binary (0 / 1) with classes 3,502 vs 3,452 — so this notebook uses `CatBoostClassifier` in binary mode. A short section at the end shows the (very small) edits needed to switch to multiclass.

> **Note on "text fields".** Your `object`-typed columns (`PromoCode`, `Region`, `PackageType`, `BookingChannel`, `AgeGroup`, `ReferralSource`) are short categorical strings, not free-text. CatBoost's dedicated `text_features` mechanism is for things like customer reviews or support tickets. For your data, those columns belong in `cat_features`. A short callout shows what `text_features` would look like if you later add a free-text column (e.g. survey comments).


## 1. Environment

CatBoost has no hard dependencies you don't already have. `shap` is needed for the interpretation section, `optuna` for the tuning section.

In [ ]:
# Run once. Comment out after the first install.
# %pip install -q catboost==1.2.7 optuna==3.6.1 shap==0.46.0 scikit-learn pandas matplotlib

In [ ]:
import warnings, json, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, f1_score,
)

import catboost as cb
from catboost import CatBoostClassifier, Pool

print("catboost version:", cb.__version__)
np.random.seed(42)

## 2. Load the data

We use the cleaned training CSV directly. `GuestID` is an identifier and must not become a feature.

In [ ]:
DATA_PATH = "/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# Class balance
df["Churned"].value_counts(normalize=True).rename("share")

## 3. Declare feature types

CatBoost performs best when you **tell it explicitly** which columns are categorical. Internally it uses *ordered target statistics* to encode them — that's where most of its accuracy advantage over LightGBM/XGBoost on tabular data comes from.

The rules for this dataset:

- **Identifier (drop):** `GuestID`
- **Target:** `Churned`
- **Date (engineer, don't pass raw):** `BookingDate`
- **Categorical (`cat_features`):** `PromoCode`, `Region`, `PackageType`, `BookingChannel`, `AgeGroup`, `ReferralSource`
- **Binary flags (already 0/1 — leave as numeric):** `AllInclusive`, `VIP`, `RoomService`, `Dining`, `Retail`, `Spa`, `Entertainment`, `SharedRoom`
- **Continuous numeric:** `LoyaltyPoints`, `SurveyScore`, `DaysSinceEmail`

> Why not also pass the binary flags as `cat_features`? Because they're already perfectly numeric (0/1). Passing them as categorical would force CatBoost to compute ordered TS for them — pure overhead with zero accuracy gain.


In [ ]:
TARGET = "Churned"

# Light date feature engineering — CatBoost won't parse a raw date string.
df["BookingDate"] = pd.to_datetime(df["BookingDate"])
df["Booking_month"]  = df["BookingDate"].dt.month
df["Booking_dow"]    = df["BookingDate"].dt.dayofweek
df["Booking_doy"]    = df["BookingDate"].dt.dayofyear
df = df.drop(columns=["BookingDate", "GuestID"])

cat_features = [
    "PromoCode", "Region", "PackageType",
    "BookingChannel", "AgeGroup", "ReferralSource",
]

# Continuous numerics we'll later target with per_float_feature_quantization
continuous_features = ["LoyaltyPoints", "SurveyScore", "DaysSinceEmail"]

# Everything else (0/1 flags + engineered date parts) just rides along as numeric
y = df[TARGET]
X = df.drop(columns=[TARGET])

# CatBoost requires categorical columns to be string or int — never float.
for c in cat_features:
    X[c] = X[c].astype(str)

X.dtypes

### Aside: what `text_features` would look like

If you later add a column like `SurveyComment` containing free-form sentences, CatBoost can tokenize and learn from it natively:

```python
text_features = ["SurveyComment"]
model = CatBoostClassifier(
    cat_features=cat_features,
    text_features=text_features,
    tokenizers=[{"tokenizer_id": "Space", "separator_type": "ByDelimiter", "delimiter": " "}],
    dictionaries=[{"dictionary_id": "Word", "max_dictionary_size": "50000"}],
    feature_calcers=["BoW", "NaiveBayes"],
)
```

For the current dataset, every string column is a short category, so we use `cat_features` only.

## 4. Train / validation split

Stratify on the target so the validation split sees the same class ratio.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print("Train churn rate:", y_train.mean().round(4))
print("Val   churn rate:", y_val.mean().round(4))

In [ ]:
# CatBoost Pools are the canonical way to pass data — they cache the categorical
# preprocessing and let you reuse the same data across multiple fit calls without
# re-encoding it each time.
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool   = Pool(X_val,   y_val,   cat_features=cat_features)

## 5. Baseline model

Goals for the baseline:

1. Confirm the pipeline runs end-to-end.
2. Get an honest validation score we can compare every later experiment against.
3. Enable **early stopping** so we never accidentally compare an overfit run to a well-fit one.

The single most important CatBoost-specific knob is `eval_metric`. Set it to what you actually care about (`AUC` for ranking quality, `Logloss` for calibration, `F1` for imbalanced classification). The training metric (`loss_function`) should usually stay as `Logloss` for binary even when `eval_metric` differs.

In [ ]:
baseline = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    od_type="Iter",          # overfitting detector
    od_wait=50,              # stop if eval metric hasn't improved in 50 rounds
    verbose=200,
)

baseline.fit(train_pool, eval_set=val_pool, use_best_model=True)
print(f"\nBest iteration: {baseline.get_best_iteration()}")
print(f"Best AUC      : {baseline.get_best_score()['validation']['AUC']:.4f}")

In [ ]:
def evaluate(model, X_eval, y_eval, name="Model"):
    proba = model.predict_proba(X_eval)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    print(f"=== {name} ===")
    print(f"AUC      : {roc_auc_score(y_eval, proba):.4f}")
    print(f"PR-AUC   : {average_precision_score(y_eval, proba):.4f}")
    print(f"F1@0.5   : {f1_score(y_eval, pred):.4f}")
    print(classification_report(y_eval, pred, digits=3))

evaluate(baseline, X_val, y_val, "Baseline")

## 6. Class imbalance handling

Your training set is essentially balanced (≈50.4% churned). You still want this code in your back pocket for two reasons:

1. The **test** distribution can drift — production churn rates change with marketing pushes, seasonality, etc.
2. Even at 50/50 you may care asymmetrically (false negatives cost more than false positives, or vice versa).

CatBoost offers three mechanisms, **mutually exclusive** — pick one:

| Option | When to use |
|---|---|
| `auto_class_weights="Balanced"` | Quickest fix. Weights inversely proportional to class frequency. |
| `auto_class_weights="SqrtBalanced"` | Softer reweighting; good when "Balanced" overcorrects and tanks AUC. |
| `class_weights=[w0, w1]` | Manual weights — use when you've quantified the cost asymmetry. |
| `scale_pos_weight=k` | Single-scalar version of `class_weights` for binary. Equivalent to `class_weights=[1, k]`. |

Below: a quick comparison.

In [ ]:
def fit_quick(**kwargs):
    m = CatBoostClassifier(
        iterations=600, learning_rate=0.05, depth=6,
        loss_function="Logloss", eval_metric="AUC",
        random_seed=42, verbose=0, od_type="Iter", od_wait=40,
        **kwargs,
    )
    m.fit(train_pool, eval_set=val_pool, use_best_model=True)
    return m

variants = {
    "no_weighting":       fit_quick(),
    "auto_balanced":      fit_quick(auto_class_weights="Balanced"),
    "auto_sqrt":          fit_quick(auto_class_weights="SqrtBalanced"),
    "scale_pos_weight=2": fit_quick(scale_pos_weight=2.0),
}

rows = []
for name, m in variants.items():
    proba = m.predict_proba(X_val)[:, 1]
    rows.append({
        "variant": name,
        "AUC":    round(roc_auc_score(y_val, proba), 4),
        "PR_AUC": round(average_precision_score(y_val, proba), 4),
        "F1@0.5": round(f1_score(y_val, (proba >= 0.5).astype(int)), 4),
    })
pd.DataFrame(rows)

On nearly-balanced data the variants typically score within noise of each other. Re-run this comparison whenever your production class ratio shifts past ~60/40.

## 7. Highlighting specific columns

This is the section you specifically asked about. CatBoost gives you **two** different ways to "tell the model some columns matter more," and they do different things. Use them together.

### 7.1 `feature_weights` — bias split selection

When CatBoost evaluates candidate splits at each tree level, it scores them with a loss-reduction heuristic. `feature_weights` *multiplies that heuristic score by `w_i`* for each feature `i`. The result is that features with `w_i > 1.0` are more likely to be chosen for splits.

**Use it when**:

- You have prior knowledge that certain features are causally important (domain knowledge)
- A baseline model is under-utilizing a feature you know should matter
- You're chasing better generalization on a feature by forcing the model to lean on it

**Don't use it to**:

- Compensate for bad data (fix the data instead)
- Force the model to use a leaky feature (it'll just overfit harder)

The argument can be:
- a list of floats in column order (`[1.0, 2.5, 1.0, ...]`)
- a dict keyed by feature name (much safer)

For churn, plausible "I know these matter" features are `LoyaltyPoints` (engaged guests churn less), `SurveyScore` (direct signal), and `DaysSinceEmail` (engagement recency).

In [ ]:
feature_weights_dict = {
    "LoyaltyPoints":  2.0,   # double the split priority
    "SurveyScore":    2.5,
    "DaysSinceEmail": 1.5,
    # all other features default to 1.0
}

model_fw = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    loss_function="Logloss", eval_metric="AUC",
    feature_weights=feature_weights_dict,
    random_seed=42, verbose=200, od_type="Iter", od_wait=50,
)
model_fw.fit(train_pool, eval_set=val_pool, use_best_model=True)
evaluate(model_fw, X_val, y_val, "feature_weights")

**How to read the result.** If AUC is *higher* than the baseline, your prior was useful. If *lower*, CatBoost's data-driven split scoring already valued those features correctly and you're now distorting it — back the weights off toward 1.0 or remove them.

> **Practical tip.** Treat `feature_weights` like a *Bayesian prior*: small deviations from 1.0 (1.2–2.5) work well. Values like 10.0 or 100.0 almost always hurt because they let weak splits on the favored feature beat better splits elsewhere.

### 7.2 `per_float_feature_quantization` — finer-grained bins for important numerics

CatBoost converts every continuous numeric feature into a **discrete set of bins** (default 254) and only considers split points at bin boundaries. This is what makes it fast.

The trade-off: if a column has a sharp, narrow region where the relationship to the target changes (e.g. `LoyaltyPoints` matters most below 500 and above 5000, but is flat in between), 254 evenly-spaced bins may not place a boundary in the right place.

`per_float_feature_quantization` lets you set **more bins** and a **different quantization strategy** for specific columns.

**Border counts.** Maximum is 65,535 but anything above ~1024 is usually waste. Try 256 → 512 → 1024 if your important numeric has a complex shape.

**Border types.**

- `GreedyLogSum` (default) — usually fine
- `Uniform` — equal-width bins; useful when values are uniformly distributed
- `UniformAndQuantiles` — hybrid; good for skewed distributions with long tails
- `MaxLogSum`, `MedianInBin`, `MinEntropy` — specialty options, rarely better

Syntax is `<feature_index_or_name>:border_count=N:border_type=TYPE`. Note CatBoost wants **feature indices**, not names, for this argument when passed as a list — we'll build it programmatically.

In [ ]:
# Map feature names to their column indices in X (the order matters)
feat_index = {name: i for i, name in enumerate(X.columns)}

per_float_q = [
    f"{feat_index['LoyaltyPoints']}:border_count=1024:border_type=UniformAndQuantiles",
    f"{feat_index['SurveyScore']}:border_count=64",   # SurveyScore is discrete 1-5 ish — high bin count is wasteful
    f"{feat_index['DaysSinceEmail']}:border_count=512:border_type=GreedyLogSum",
]
print(per_float_q)

In [ ]:
model_pfq = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    loss_function="Logloss", eval_metric="AUC",
    per_float_feature_quantization=per_float_q,
    random_seed=42, verbose=200, od_type="Iter", od_wait=50,
)
model_pfq.fit(train_pool, eval_set=val_pool, use_best_model=True)
evaluate(model_pfq, X_val, y_val, "per_float_feature_quantization")

### 7.3 Combine both

The two mechanisms are independent and compose cleanly. `feature_weights` controls *which feature gets picked*; `per_float_feature_quantization` controls *where the splits on that feature can be placed*.

In [ ]:
model_combo = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    loss_function="Logloss", eval_metric="AUC",
    feature_weights=feature_weights_dict,
    per_float_feature_quantization=per_float_q,
    random_seed=42, verbose=200, od_type="Iter", od_wait=50,
)
model_combo.fit(train_pool, eval_set=val_pool, use_best_model=True)
evaluate(model_combo, X_val, y_val, "fw + pfq")

In [ ]:
# Comparison table
def auc(m): return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
pd.DataFrame([
    {"model": "baseline",                   "AUC": round(auc(baseline),    4)},
    {"model": "+ feature_weights",          "AUC": round(auc(model_fw),    4)},
    {"model": "+ per_float_quantization",   "AUC": round(auc(model_pfq),   4)},
    {"model": "+ both",                     "AUC": round(auc(model_combo), 4)},
])

## 8. Hyperparameter tuning — full Optuna deep-dive

For a model this size on CPU, an Optuna study of 50–100 trials typically lands within 0.1–0.3 AUC points of what an unlimited search would find. The knobs that matter, ranked by typical impact:

1. **`depth`** (3–10) — tree depth. Higher = more interactions, more overfitting risk.
2. **`learning_rate`** (0.01–0.3) — paired with `iterations`. Lower LR + more iterations is usually better but slower.
3. **`l2_leaf_reg`** (1–10) — L2 regularization on leaf values. Bumping this is the cheapest defense against overfitting.
4. **`bagging_temperature`** (0–10) — Bayesian bagging temperature. 0 = no bagging, ∞ = uniform weights.
5. **`random_strength`** (0–10) — noise injected into split scores. Higher = more diverse trees.
6. **`border_count`** (32–255) — number of bins for *all* numeric features. Per-feature overrides win.
7. **`min_data_in_leaf`** (1–100) — minimum samples per leaf. Larger = more regularization.

We optimize AUC on a held-out validation slice. Use a stratified 5-fold CV inside the objective for more stable scores — the example below uses single-split for speed.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        iterations=2000,
        learning_rate=trial.suggest_float("learning_rate", 1e-2, 3e-1, log=True),
        depth=trial.suggest_int("depth", 4, 10),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 5.0),
        random_strength=trial.suggest_float("random_strength", 1e-6, 10.0, log=True),
        border_count=trial.suggest_int("border_count", 32, 254),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
        # The "highlight columns" settings stay fixed across trials —
        # we already chose them based on domain knowledge.
        feature_weights=feature_weights_dict,
        per_float_feature_quantization=per_float_q,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        od_type="Iter",
        od_wait=40,
    )
    m = CatBoostClassifier(**params)
    m.fit(train_pool, eval_set=val_pool, use_best_model=True)
    return m.get_best_score()["validation"]["AUC"]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
# Bump n_trials up to 100+ for serious tuning. 30 is enough to demonstrate.
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC :", round(study.best_value, 4))
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Optuna importance plot — which params actually moved the needle
import optuna.importance as oi
imp = oi.get_param_importances(study)
pd.DataFrame(imp.items(), columns=["param", "importance"]).round(3)

### Refit the final model with best params + early stopping headroom

Use a higher `iterations` cap because we now trust early stopping to find the right number.

In [ ]:
best_params = dict(study.best_params)
final_model = CatBoostClassifier(
    iterations=5000,
    **best_params,
    feature_weights=feature_weights_dict,
    per_float_feature_quantization=per_float_q,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    od_type="Iter",
    od_wait=100,    # give early stopping more patience for the final fit
)
final_model.fit(train_pool, eval_set=val_pool, use_best_model=True)

print("Best iter :", final_model.get_best_iteration())
evaluate(final_model, X_val, y_val, "Final tuned")

### Cross-validated final check

Single-split AUC is noisy at this dataset size. Confirm the tuned config holds up under 5-fold CV.

In [ ]:
from catboost import cv as cb_cv

cv_params = {
    **best_params,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": int(final_model.get_best_iteration() * 1.1) + 1,  # small buffer
}

cv_results = cb_cv(
    pool=Pool(X, y, cat_features=cat_features),
    params=cv_params,
    fold_count=5,
    stratified=True,
    seed=42,
    verbose=False,
    plot=False,
)
print(f"5-fold AUC: {cv_results['test-AUC-mean'].iloc[-1]:.4f} "
      f"± {cv_results['test-AUC-std'].iloc[-1]:.4f}")

## 9. SHAP — verify `feature_weights` did what you intended

This is the section many tutorials skip and it's the most important one. The whole point of `feature_weights` is to push the model toward the columns *you* care about. SHAP tells you whether that actually happened.

CatBoost ships with native SHAP value computation — it's exact and fast, no Kernel/Permutation approximation needed.

In [ ]:
import shap

# Native CatBoost SHAP — returns shape (n_rows, n_features + 1); last column is bias.
shap_values_full = final_model.get_feature_importance(
    val_pool, type="ShapValues"
)
shap_vals = shap_values_full[:, :-1]
expected_value = shap_values_full[0, -1]
print("Shape:", shap_vals.shape, "  expected value:", expected_value)

In [ ]:
# Global importance — mean absolute SHAP per feature
mean_abs = np.abs(shap_vals).mean(axis=0)
imp_df = (
    pd.DataFrame({"feature": X.columns, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
imp_df.head(15)

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
top = imp_df.head(15).iloc[::-1]
ax.barh(top["feature"], top["mean_abs_shap"])
ax.set_xlabel("Mean |SHAP|")
ax.set_title("Top 15 features (CatBoost native SHAP)")
plt.tight_layout()
plt.show()

**Sanity check for `feature_weights`.**

Look at where `LoyaltyPoints`, `SurveyScore`, `DaysSinceEmail` land in the importance ranking compared to the baseline. If they moved up, `feature_weights` is doing its job. If they didn't, either the weights were too small (try 3–5) or the prior was just wrong — those features really aren't predictive on this data and you should remove the weighting.

In [ ]:
# Side-by-side: baseline importance vs. weighted-model importance
def shap_importance(model, pool, columns):
    vals = model.get_feature_importance(pool, type="ShapValues")[:, :-1]
    return pd.Series(np.abs(vals).mean(axis=0), index=columns)

base_imp  = shap_importance(baseline,    val_pool, X.columns).rename("baseline")
final_imp = shap_importance(final_model, val_pool, X.columns).rename("tuned+weighted")

side = pd.concat([base_imp, final_imp], axis=1)
side["delta"] = side["tuned+weighted"] - side["baseline"]
side = side.sort_values("tuned+weighted", ascending=False)
side.head(15).round(4)

### Per-prediction explanation

You'll be asked "*why did the model flag this guest?*". SHAP answers that.

In [ ]:
# Pick a guest the model thinks is high-risk
proba = final_model.predict_proba(X_val)[:, 1]
i = int(np.argmax(proba))
print(f"Validation row {i}: churn probability {proba[i]:.3f}")

per_row = pd.DataFrame({
    "feature": X.columns,
    "value":   X_val.iloc[i].values,
    "shap":    shap_vals[i],
}).sort_values("shap", key=lambda s: s.abs(), ascending=False)
per_row.head(10)

## 10. Save, load, predict

CatBoost has two on-disk formats: native `.cbm` (fast, complete) and `.json` (portable, larger). Use `.cbm` for production unless you need cross-language interoperability.

In [ ]:
MODEL_PATH = "/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_catboost_final.cbm"
final_model.save_model(MODEL_PATH)
print("Saved to:", MODEL_PATH)

In [ ]:
# Reload and confirm parity
reloaded = CatBoostClassifier()
reloaded.load_model(MODEL_PATH)
np.testing.assert_allclose(
    final_model.predict_proba(X_val)[:, 1],
    reloaded.predict_proba(X_val)[:, 1],
    atol=1e-9,
)
print("Reload OK — identical predictions.")

## 11. Extending to multiclass (short note)

If you later have a multi-class target (`{stayed, downgraded, churned, escalated}` etc.), the only changes are:

```python
model = CatBoostClassifier(
    loss_function="MultiClass",          # was: Logloss
    eval_metric="MultiClass",            # or "TotalF1", "Accuracy"
    classes_count=4,                     # optional — autodetected
    # everything else stays the same — feature_weights, per_float_feature_quantization, etc.
)
```

For class imbalance use `class_weights={cls: w}` as a dict; `scale_pos_weight` is binary-only.

`predict_proba` returns shape `(n_rows, n_classes)`. SHAP returns one set of values *per class* — slice accordingly.

## 12. Checklist for a production-ready run

Before shipping the tuned model:

1. **Re-run on full train + test** — once you trust the validation result, retrain on all available labeled data (no holdout) using the best iteration found above (no `eval_set`, set `iterations` to the value `final_model.get_best_iteration()` printed).
2. **Calibrate probabilities** if downstream consumers thresholding the score care about absolute probabilities — `sklearn.calibration.CalibratedClassifierCV(method="isotonic")` wraps the CatBoost model.
3. **Lock the schema** — `cat_features`, the date-engineering function, and the column order must match exactly at inference time. Save the column list next to the model.
4. **Monitor drift** — track input feature distributions and the predicted-probability histogram. Either drifting is your earliest warning that the model needs retraining.
5. **Re-evaluate `feature_weights`** — your priors are time-bound. What was important last quarter may not be this quarter.

That's the full loop. Adjust the Optuna trial count, the weight values, and the quantization bin counts to your dataset's quirks.